# pandas 연습 1: 추출 결과 집계

## 목표
이 프로젝트의 실제 추출 결과(청크별 `*.result.json`)로 pandas 기본기(표 만들기, 묶어서 세기, 합치기, 정렬)를 익힌다.

## 준비
- 커널은 **프로젝트 uv 환경**(`.venv`)을 선택한다.
- 정답 코드는 없다. `# TODO` 셀을 직접 채운다. 막히면 힌트의 함수 이름을 pandas 공식 문서에서 찾아본다.
- 결과 파일의 본문(청크 텍스트, 인용문)은 출력하지 않는다. 구조, 건수, 길이만 본다.
- **저장·커밋 전에 반드시 `Clear All Outputs`** 를 실행해 출력을 비운다.

In [ ]:
from pathlib import Path
import json
import pandas as pd

RESULT_DIR = Path.home() / 'pilot_work' / 'out9_hana50'   # 이번 결과
OLD_DIR = Path.home() / 'pilot_work' / 'out8_hana50'      # 비교할 이전 결과
# 청크 하나당 파일 하나: <문서해시>-pNNNN-cNN.result.json
result_files = sorted(RESULT_DIR.glob('*.result.json'))
len(result_files)

## 1. 결과 파일 하나의 키 구조 훑기
아래 셀은 주어진 코드다. 파일 하나를 열어 **최상위 키 이름만** 본다. 값(본문)은 출력하지 않는다.
키 이름 중 리스트(`[]`)로 보이는 항목이 이후 과제에서 "항목 수"를 셀 대상이다. 리스트인지 여부는 `type(값)`, 길이는 `len(값)`으로 확인한다.

In [ ]:
with open(result_files[0], encoding='utf-8') as f:
    sample = json.load(f)
print(list(sample.keys()))
# 값의 종류와 길이만 보기 (본문은 출력하지 않음)
for k, v in sample.items():
    print(k, type(v).__name__, len(v) if hasattr(v, '__len__') else '')

## 과제 1. 청크별 항목 수 표 만들기

**목표**: 청크 한 개를 한 행으로 하는 DataFrame을 만든다. 열 예: 청크 파일명, 문서 해시, 항목 종류별 개수.

**힌트(함수 이름)**
- `pd.DataFrame(list_of_dicts)`: 딕셔너리 리스트를 표로 바꾼다. 딕셔너리 하나가 한 행이 된다.
- `pd.json_normalize`: 중첩된 JSON을 평평한 열로 펼쳐 표로 만든다.
- `Series.str.split` / `Series.str.extract`: 문자열 열을 쪼개거나 정규식으로 일부를 뽑는다(파일명에서 문서 해시 추출 등).
- `groupby(...).size()` / `groupby(...).agg`: 같은 값끼리 묶어 행 수나 합계를 구한다.
- `df.describe()`: 숫자 열의 평균, 최소, 최대를 한 번에 본다.

**확인 방법**: 표의 행 수가 **50** 이어야 한다(out9는 50청크). 문서 해시별로 묶어 합친 청크 수의 총합도 50이어야 한다.

In [ ]:
# TODO: result_files를 돌며 청크별 항목 수를 모아 DataFrame `df_chunks`를 만든다

In [ ]:
# TODO: 확인 - 행 수와 문서 해시별 청크 수 합계를 출력한다

## 과제 2. out8 vs out9 빈 결과 비교

**목표**: 같은 청크가 out8에서는 비었는데 out9에서는 채워졌는지(또는 반대인지) 청크 단위로 비교한다. "빈 결과"의 기준(예: 항목 수 0)은 1번 구조를 보고 직접 정한다.

**힌트(함수 이름)**
- 과제 1의 표 만드는 코드를 함수로 묶어 `OLD_DIR` 에도 적용한다(`def`).
- `merge(on=..., how='outer', suffixes=...)`: 청크 파일명을 기준으로 두 표를 옆으로 붙인다.
- `df[조건]`: 조건이 참인 행만 고른다(불리언 인덱싱).
- `value_counts()`: 값별로 몇 개인지 센다.
- `pd.crosstab`: 두 범주(예: out8 빈 여부 x out9 빈 여부)의 교차 건수표를 만든다.

**확인 방법**: out9의 빈 결과는 **19** 개여야 한다. 교차표에서 out9 빈 칸의 합이 19인지 본다. 한쪽에만 있는 청크가 있으면 `how='outer'` 결과의 결측(NaN)으로 드러난다.

In [ ]:
# TODO: OLD_DIR 결과도 같은 방식으로 표 `df_old`를 만든다

In [ ]:
# TODO: 두 표를 merge하고 빈 결과 여부 교차표를 만든다. out9 빈 결과 개수가 19인지 확인

## 과제 3. 키워드 신호 수로 정렬해 상위 N개 뽑기

**목표**: 청크별 "키워드 신호" 개수를 열로 만들고, 많은 순으로 정렬해 상위 N개(예: 10개) 청크의 파일명과 개수만 본다. 어떤 키가 키워드 신호인지는 1번 구조를 보고 정한다. 없으면 가장 비슷한 리스트 키로 대체하고 이유를 주석으로 적는다.

**힌트(함수 이름)**
- `df['새열'] = df['열'].apply(len)`: 열의 각 값에 함수를 적용해 새 열을 만든다(리스트 길이 등).
- `sort_values(by=..., ascending=False)`: 지정한 열 기준으로 정렬한다. `False`면 큰 값부터.
- `head(N)` / `nlargest(N, '열')`: 위에서 N개만 고른다.
- `reset_index(drop=True)`: 정렬 후 흐트러진 행 번호를 0부터 다시 매긴다.
- `value_counts()`: 신호 수 값의 분포를 본다.

**확인 방법**: 결과가 N행이고, 첫 행의 값이 전체 최댓값(`df['열'].max()`)과 같아야 한다. 신호 수 합계가 과제 1의 해당 열 합계와 같은지도 비교한다.

In [ ]:
# TODO: 키워드 신호 수 열을 만들고 내림차순 정렬 후 상위 N개를 `top_n`에 담는다

In [ ]:
# TODO: 확인 - 첫 행 값과 max 비교, 합계 비교

## 마무리: 이해 확인 질문
1. 과제 1에서 한 행이 무엇을 뜻하도록 표를 만들었는가? 문서 단위로 바꾸려면 어떤 함수로 어떻게 묶겠는가?
2. 과제 2에서 `merge`의 `how='inner'` 와 `how='outer'` 는 결과가 어떻게 다른가? 청크가 한쪽에만 있을 때 어느 쪽이 그 사실을 보여 주는가?
3. `sort_values` 뒤에 `reset_index(drop=True)` 를 쓰지 않으면 어떤 점이 달라지는가? 원본 `df`는 바뀌는가, 새 객체가 반환되는가?

끝나면 **Clear All Outputs** 후 저장한다.